# Pretraitement Pour L'Entrainement AQUA-ATMOS

Ce notebook prepare un jeu d'entree propre, documente et reutilisable pour la comparaison de modeles.


## 1. Chargement

**Objectif**
Charger le jeu synthetique principal et verifier qu'il contient bien les colonnes necessaires.

**Resultat attendu**
Disposer d'un DataFrame de depart pret pour les transformations.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SYNTHETIC_PATH = PROJECT_ROOT / "data" / "synthetic" / "synthetic_year.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SYNTHETIC_PATH)
df.head()


,profile_name,day_index,hour,temp_air_c,hr_pct,solar_wm2,pv_voltage,temp_collector_c,temp_cond_c,delta_hr_sorbent,reservoir_level_pct,soc_battery_pct,dew_point_c,humidity_ratio_gkg,vcrc_state,vcrc_reason,sorbent_mode,heater_on,sorbent_saturated
0,dakar_coastal,0,0,18.42,67.72,0.0,0.0,18.26,13.90,2.39,11.125484,70.577072,12.343953,8.929718,1,eligible,absorption,False,False
1,dakar_coastal,0,1,17.25,71.46,0.0,0.0,16.69,13.80,2.72,12.518850,63.882427,12.041524,8.750818,1,eligible,absorption,False,False
2,dakar_coastal,0,2,16.33,71.10,0.0,0.0,17.25,12.65,1.31,13.717395,56.688926,11.081749,8.205504,1,eligible,absorption,False,False
3,dakar_coastal,0,3,16.70,73.81,0.0,0.0,16.78,14.22,1.12,14.905522,49.585642,12.003688,8.728459,1,eligible,absorption,False,False
4,dakar_coastal,0,4,16.96,72.79,0.0,0.0,17.37,13.16,0.11,16.098170,42.937683,12.042814,8.751423,1,eligible,regeneration,True,True


## 2. Controle De Qualite

**Objectif**
Controler les types, doublons et valeurs manquantes avant toute transformation.

**Resultat attendu**
Identifier tout point bloquant avant l'entrainement.


In [ ]:
quality_report = {
    "shape": df.shape,
    "missing_values": df.isna().sum().to_dict(),
    "duplicates": int(df.duplicated().sum()),
}

quality_report

{'shape': (8760, 19),
 'missing_values': {'profile_name': 0,
  'day_index': 0,
  'hour': 0,
  'temp_air_c': 0,
  'hr_pct': 0,
  'solar_wm2': 0,
  'pv_voltage': 0,
  'temp_collector_c': 0,
  'temp_cond_c': 0,
  'delta_hr_sorbent': 0,
  'reservoir_level_pct': 0,
  'soc_battery_pct': 0,
  'dew_point_c': 0,
  'humidity_ratio_gkg': 0,
  'vcrc_state': 0,
  'vcrc_reason': 0,
  'sorbent_mode': 0,
  'heater_on': 0,
  'sorbent_saturated': 0},
 'duplicates': 0}

## 3. Normalisation Des Noms De Profils

**Objectif**
Rapprocher les anciens identifiants synthetiques des identifiants de villes standardises.

**Resultat attendu**
Obtenir une colonne `city_id` stable pour les futurs modeles et dashboards.


In [3]:
city_aliases = {
    "agadir_coastal": "agadir",
    "dakar_coastal": "dakar",
    "abidjan_coastal": "abidjan",
    "douala_coastal": "douala",
    "mombasa_coastal": "mombasa",
    "walvis_bay_coastal": "walvis_bay",
}

df["city_id"] = df["profile_name"].replace(city_aliases)
df[["profile_name", "city_id"]].drop_duplicates().sort_values(["city_id", "profile_name"])


,profile_name,city_id
0,dakar_coastal,dakar


## 4. Variables Temporelles Cycliques

**Objectif**
Encoder les cycles jour/nuit et saisonniers sans casser la periodicite.

**Resultat attendu**
Ajouter des variables robustes pour les modeles lineaires et arborescents.


In [4]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["day_sin"] = np.sin(2 * np.pi * df["day_index"] / 365)
df["day_cos"] = np.cos(2 * np.pi * df["day_index"] / 365)

df[["hour", "hour_sin", "hour_cos", "day_index", "day_sin", "day_cos"]].head()


,hour,hour_sin,hour_cos,day_index,day_sin,day_cos
0,0,0.000000,1.000000,0,0.0,1.0
1,1,0.258819,0.965926,0,0.0,1.0
2,2,0.500000,0.866025,0,0.0,1.0
3,3,0.707107,0.707107,0,0.0,1.0
4,4,0.866025,0.500000,0,0.0,1.0


## 5. Variables Derivees Metier

**Objectif**
Creer quelques indicateurs interpretablement utiles pour l'entrainement.

**Resultat attendu**
Enrichir le jeu sans dupliquer inutilement les informations deja presentes.


In [5]:
df["thermal_lift_c"] = df["temp_air_c"] - df["temp_cond_c"]
df["collector_gain_c"] = df["temp_collector_c"] - df["temp_air_c"]
df["is_daylight"] = (df["solar_wm2"] > 0).astype(int)
df["high_humidity_flag"] = (df["hr_pct"] >= 70).astype(int)
df["battery_stress_flag"] = (df["soc_battery_pct"] <= 25).astype(int)
df["reservoir_high_flag"] = (df["reservoir_level_pct"] >= 80).astype(int)

engineered_columns = [
    "thermal_lift_c",
    "collector_gain_c",
    "is_daylight",
    "high_humidity_flag",
    "battery_stress_flag",
    "reservoir_high_flag",
]
df[engineered_columns].describe().T


,count,mean,std,min,25%,50%,75%,max
thermal_lift_c,8760.0,2.724155,0.700377,1.36,2.2,2.69,3.14,6.1
collector_gain_c,8760.0,5.226100,6.599823,-1.00,-0.1,0.85,10.80,21.7
is_daylight,8760.0,0.458333,0.498289,0.00,0.0,0.00,1.00,1.0
high_humidity_flag,8760.0,0.435160,0.495806,0.00,0.0,0.00,1.00,1.0
battery_stress_flag,8760.0,0.894863,0.306747,0.00,1.0,1.00,1.00,1.0
reservoir_high_flag,8760.0,0.251027,0.433629,0.00,0.0,0.00,1.00,1.0


## 6. Encodage Des Cibles

**Objectif**
Preparer des cibles numeriques stables pour les taches de classification.

**Resultat attendu**
Obtenir des labels exploitables pour les comparaisons de modeles.


In [6]:
sorbent_mode_mapping = {
    "veille": 0,
    "absorption": 1,
    "regeneration": 2,
}

df["sorbent_mode_label"] = df["sorbent_mode"].map(sorbent_mode_mapping)
df["heater_on_label"] = df["heater_on"].astype(int)
df["sorbent_saturated_label"] = df["sorbent_saturated"].astype(int)

df[["sorbent_mode", "sorbent_mode_label", "heater_on_label", "sorbent_saturated_label"]].head()


,sorbent_mode,sorbent_mode_label,heater_on_label,sorbent_saturated_label
0,absorption,1,0,0
1,absorption,1,0,0
2,absorption,1,0,0
3,absorption,1,0,0
4,regeneration,2,1,1


## 7. Colonnes D'Entree Proposees

**Objectif**
Definir explicitement la table des features de reference pour la phase de benchmark.

**Resultat attendu**
Avoir une liste de colonnes stable et auditable.


In [7]:
feature_columns = [
    "temp_air_c",
    "hr_pct",
    "solar_wm2",
    "pv_voltage",
    "temp_collector_c",
    "temp_cond_c",
    "delta_hr_sorbent",
    "reservoir_level_pct",
    "soc_battery_pct",
    "dew_point_c",
    "humidity_ratio_gkg",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "thermal_lift_c",
    "collector_gain_c",
    "is_daylight",
    "high_humidity_flag",
    "battery_stress_flag",
    "reservoir_high_flag",
]

target_columns = [
    "vcrc_state",
    "sorbent_mode_label",
    "heater_on_label",
    "sorbent_saturated_label",
]

pd.DataFrame(
    {
        "feature_columns": pd.Series(feature_columns),
        "target_columns": pd.Series(target_columns),
    }
)


,feature_columns,target_columns
0,temp_air_c,vcrc_state
1,hr_pct,sorbent_mode_label
2,solar_wm2,heater_on_label
3,pv_voltage,sorbent_saturated_label
4,temp_collector_c,NaN
5,temp_cond_c,NaN
6,delta_hr_sorbent,NaN
7,reservoir_level_pct,NaN
8,soc_battery_pct,NaN
9,dew_point_c,NaN


## 8. Partition Sans Fuite

**Objectif**
Construire une separation `train / validation / test` par blocs temporels.

**Resultat attendu**
Eviter qu'un modele voie indirectement le futur pendant l'apprentissage.


In [8]:
df = df.sort_values(["city_id", "day_index", "hour"]).reset_index(drop=True)

train_mask = df["day_index"] < 255
valid_mask = (df["day_index"] >= 255) & (df["day_index"] < 310)
test_mask = df["day_index"] >= 310

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [int(train_mask.sum()), int(valid_mask.sum()), int(test_mask.sum())],
        "day_index_min": [
            int(df.loc[train_mask, "day_index"].min()),
            int(df.loc[valid_mask, "day_index"].min()),
            int(df.loc[test_mask, "day_index"].min()),
        ],
        "day_index_max": [
            int(df.loc[train_mask, "day_index"].max()),
            int(df.loc[valid_mask, "day_index"].max()),
            int(df.loc[test_mask, "day_index"].max()),
        ],
    }
)

split_summary


,split,rows,day_index_min,day_index_max
0,train,6120,0,254
1,validation,1320,255,309
2,test,1320,310,364


## 9. Export Des Jeux Prepares

**Objectif**
Sauvegarder des tables reutilisables pour les notebooks d'entrainement.

**Resultat attendu**
Produire des fichiers CSV clairs, versionnables et lisibles sans code supplementaire.


In [9]:
prepared_df = df[["city_id", "day_index", "hour", *feature_columns, *target_columns]].copy()

train_df = prepared_df.loc[train_mask].copy()
valid_df = prepared_df.loc[valid_mask].copy()
test_df = prepared_df.loc[test_mask].copy()

prepared_path = OUTPUT_DIR / "training_dataset_prepared.csv"
train_path = OUTPUT_DIR / "train_split.csv"
valid_path = OUTPUT_DIR / "validation_split.csv"
test_path = OUTPUT_DIR / "test_split.csv"

prepared_df.to_csv(prepared_path, index=False)
train_df.to_csv(train_path, index=False)
valid_df.to_csv(valid_path, index=False)
test_df.to_csv(test_path, index=False)

pd.DataFrame(
    {
        "file": [prepared_path.name, train_path.name, valid_path.name, test_path.name],
        "rows": [len(prepared_df), len(train_df), len(valid_df), len(test_df)],
    }
)


,file,rows
0,training_dataset_prepared.csv,8760
1,train_split.csv,6120
2,validation_split.csv,1320
3,test_split.csv,1320


## 10. Regles Pour Le Notebook D'Entrainement

**Objectif**
Fixer les bonnes pratiques pour la prochaine etape de benchmark de modeles.

**Resultat attendu**
Une ligne directrice simple avant d'ecrire le notebook d'entrainement.


In [10]:
training_guidelines = [
    "Comparer plusieurs modeles sur exactement les memes splits.",
    "Faire le fit des scalers et encodeurs uniquement sur le train.",
    "Conserver validation pour le choix des hyperparametres, test pour l'evaluation finale.",
    "Reporter precision, recall, F1 et matrice de confusion par cible.",
    "Garder un suivi separe des cas extremes pour verifier la robustesse.",
]

pd.DataFrame({"guideline": training_guidelines})


,guideline
0,Comparer plusieurs modeles sur exactement les ...
1,Faire le fit des scalers et encodeurs uniqueme...
2,Conserver validation pour le choix des hyperpa...
3,"Reporter precision, recall, F1 et matrice de c..."
4,Garder un suivi separe des cas extremes pour v...
